# CAD Predictor Analysis Using Non-Parametric Methods

## Overview

This project analyzes the UCI Heart Disease Dataset, specifically from the Cleveland database. In this notebook, I aim to identify the top predictors of heart disease severity out of the 13 features collected by the UCI researchers and use those features to train an accurate predictive machine learning algorithm. As a long-time resident of Northeast Ohio and aspiring bioinformaticist, I am very passionate about this project, and plan to make additions in the future.

### This project focuses on:
- Exploratory data analysis (EDA)  
- Hypothesis testing via group comparison
- Effect size 
- Bootstrapped confidence intervals
- Predictive machine learning

### Primary statistical methods included:
- Spearman correlation
- Mann–Whitney U test
- Kruskal–Wallis test
- Rank-biserial correlation
- Cohen's effect size
- Bootstrap resampling

Note:  
For the purposes of readability, floats are often truncated to three decimal places throughout this report. Although Python automatically drops trailing zeros, every float in this report has at minimum three significant figures.

# Imports

In [ ]:
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
import seaborn as sb
from scipy import stats
import forestplot as fp
import math
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from scikit_posthocs import posthoc_dunn

from ucimlrepo import fetch_ucirepo 



# Loading the Dataset

In [ ]:
heart_disease = fetch_ucirepo(id=45) 

x = heart_disease.data.features 
y = heart_disease.data.targets 

xy = pd.concat([x,y], axis=1)

# Data Cleaning

In [ ]:
missing = x.isna().sum()
print("# of missing values = " + str(missing.sum()))

xy = pd.concat([x,y], axis=1)
xy = xy.dropna(how='any',axis=0, ignore_index = True)

data_points = 297

X = xy.iloc[:, :13]
y = xy.iloc[:, 13:]

Because a quick overview of the data revealed there to be at most 6 rows with missing data, it was deemed unlikely that removing them would have substantial impact on the results of the analysis, so listwise deletion was employed to clean the data.

The dataset initially contained 303 observations, which was reduced to 297 after this procedure.

# Dataset Overview

In [ ]:
xy.info()
xy.describe()

# Exploratory Data Analysis

#### CAD Demographic Information

In [ ]:
graphable = xy["num"].value_counts()
ax = graphable.plot(kind="bar", rot = 0)
for container in ax.containers:
    ax.bar_label(container = container)
plt.xlabel("Severity of CAD")
plt.ylabel("Number of patients")
plt.title("CAD Demographic Information")
plt.show()

This is the basic information on the occurence of each level of CAD severity in the dataset. Specifically, it is important to call attention to the relatively few cases of "level 4" CAD and the high percentage of asymptomatic patients.

#### Correlation Heatmap

In [ ]:
corr_matrix = xy.corr(method = "spearman")

plt.figure(figsize=(8, 6))
sb.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title("Correlation Heatmap")
plt.show()

As shown by the figure, the numerical features most strongly associated with CAD severity (num), are oldpeak, maximum heart rate (thalach), and # major vessels blocked (ca). The first two have more posssibility for variance than the latter, so we will analyze the distributions of those using a box-and-whisker plot.

It's worth noting that some of the values in this figure are meaningless. Cp, short for chest pain, for example, is a variable where '4' denotes 'asymptomatic' while 1, 2, and 3, denote various types of chest pain (further details later). As such, Spearman &rho; is incapable of capturing the variance between individuals with different types of chest pain.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

xy.boxplot(column = "thalach", by = "num", grid = False, ax=ax1)
ax1.set_xlabel("CAD severity")
ax1.set_ylabel("Max Heart Rate")
plt.suptitle("")
ax1.set_title("Max Heart Rate Grouped By CAD Severity")


xy.boxplot(column = "oldpeak", by = "num", grid = False, ax=ax2)
ax2.set_xlabel("CAD severity")
ax2.set_ylabel("Oldpeak")
plt.suptitle("")
ax2.set_title("Oldpeak Grouped By CAD Severity")


plt.tight_layout()
plt.show()


Where oldpeak is defined as the magnitude of ST-segment shortening induced by exercise, in mm, indicating that some cardiac muscles have limited oxygen supply. 

Visually, there appears to be a downwards trend in maximum heart rate as CAD severity increases, and an upwards trend in oldpeak. Tukey's HSD testing reveals that, in fact, the pairs for which we reject the null hypothesis are exactly those which include asymptomatic patients on the left, and are exactly pairs which include group 1 or group 0 (but not both) on the right.

In particular, the increase in median of maximum heart rate between groups 3 and 4 is likely due to the small sample size, as can be revealed through bootstrapping Cohen's effect size, shown below:

In [ ]:
helper = []
for i in range(1000):
    sample = xy.sample(frac=1, replace=True)
    g0 = sample.loc[sample["num"] == 3, "thalach"]
    g1 = sample.loc[sample["num"] == 4, "thalach"]
    n0,n1 = len(g0), len(g1)
    cohen_d = g1.mean() - g0.mean()
    pooled_std = math.sqrt(((n1-1)*g1.std()**2 + (n0-1)*g0.std()**2)/(n0+n1-2))
    cohen_d = cohen_d/pooled_std
    helper.append(cohen_d)

helper_ser = pd.Series(helper)

ci_lower = helper_ser.quantile(0.025)
ci_upper = helper_ser.quantile(0.975)

print("Bootstrapped Cohen_d 3 vs. 4 =", str((round(ci_lower, 3), round(ci_upper, 3))))


Since the interval contains 0, it remains possible that the correlation works in the opposite way as depicted, which is what one might expect.  

#### First look at exercise-induced angina

In [ ]:
g0 = xy.loc[xy["exang"] == 0, "num"]
g1 = xy.loc[xy["exang"] == 1, "num"]

ga = g0.value_counts(normalize=True)
gb = g1.value_counts(normalize=True)
gb = gb.sort_index()

ggs = pd.concat([ga ,gb], axis = 1)
ggs.columns = ['No Exang', 'Exang']

ggs.T.plot.bar(stacked = True, colormap = "YlOrRd", rot = 360)
plt.show()

Exercise-induced angina (exang), defined as a feeling of tightness or pain in the chest following exercise, is very strongly correlated with the presence of CAD. As shown above, nearly 70% of patients in the dataset who do not experience exang are asymptomatic for CAD, compared to only about 20% who do experience it. 

Similar to oldpeak, exang often results from parts of the heart not recieving enough oxygen-rich blood during exercise. Thus, it is unsurprising that CAD is similarly positively correlated with exang.

# Numerical Risk Factors w/ Non-Parametric Methods

A few numerical variables, particularly the number of major vessels blocked, maximum heart rate, and oldpeak were strong predictors of CAD severity. Because all of these features take on gradient, numerical values, the Spearman &rho; coefficient is sufficient to identify the strength of correlation between CAD severity and each one, as shown below.

In [ ]:
noncategorical =  ["thalach", "oldpeak", "ca"]

for i in noncategorical:
    spearman_corr = xy[[i, 'num']].corr(method = 'spearman').iloc[0,1]
    print(f"Spearman corr for {i} = {round(spearman_corr, 3)}")

### Bootstrapping Confidence Intervals

Like earlier, we can't yet be certain that these results are reflected in the larger population. In order to avoid errors that come from a relatively small sample size, we employ bootstraped confidence intervals, using the following function:

In [ ]:
def bootstrap_spearman(feature):
    helper = []
    for j in range(1000):
        sample = xy.sample(frac=1, replace = True)
        plottable = sample[[feature, 'num']]
        helper.append((plottable.corr(method = "spearman")).iloc[0, 1])
    helper_series = pd.Series(helper)
    ci_lower = helper_series.quantile(0.025)
    ci_upper = helper_series.quantile(0.975)
    return (ci_lower, ci_upper, helper_series)

We can be 95% sure that, assuming random sampling, the true Spearman &rho; value for the total population is between the two returned floats. We compute these intervals for the three top numerical predictors below:

In [ ]:
for i in noncategorical:
    lower, upper, b = bootstrap_spearman(i) #Helper_series will become useful later
    print(f"Bootstrap CI for {i} is {(float(round(lower, 3)), float(round(upper, 3)))}")

Notably, none of these intervals contain 0. In fact, each interval represents at worst a moderate correlation.Thus, we can be 95% sure that in the actual population, these variables are moderate to strong predictors of CAD and CAD severity, making them important risk factors. To demonstrate how exactly this works, we prepared a histogram showing the distribution of Spearman &rho; during the many resamplings:


In [ ]:
lower, upper, bootstraped_vals = bootstrap_spearman('ca')

plt.hist(bootstraped_vals, edgecolor = "black")
plt.xlabel('Spearman \u03C1 coefficients in bootstrapped samples')
plt.ylabel('Number of occurences (n=1000)')
plt.title("Distribution of Spearman \u03C1 in Bootstrapped Samples (ca)")
plt.axvline(x=lower, color = "black")
plt.axvline(x=upper, color = "black")
plt.show()

### Plotting Top Numerical Predictors

Now, we graph these CIs in a forest plot below:

In [ ]:
ncatvals = pd.DataFrame(columns = ["label", 'mid', 'll', 'hl'], index=noncategorical)

def build_results_ncat(label, feature, row):
    (lower, upper, b) = bootstrap_spearman(feature)
    middle = xy[[feature, 'num']].corr(method="spearman").iloc[0,1]
    ncatvals.iloc[row] = [label, middle, lower, upper]

build_results_ncat('Max Heart Rate', 'thalach', 0)
build_results_ncat('Oldpeak', 'oldpeak', 1)
build_results_ncat('# Major Vessels Blocked', 'ca', 2)
ncatvals.sort_values('mid')

fp.forestplot(dataframe=ncatvals, estimate = 'mid', ll = 'll', hl = 'hl', varlabel = 'label', xlim = (-0.7, 0.7), 
              xlabel = "Spearman \u03C1", figsize= (6,3), ci_report=False, form_ci_report=False,
              **{"marker": "D", "markersize": 40, "color": "black"})
plt.tight_layout()
plt.axvline(x=0,color="gray")
plt.show()

By absolute values, we see that the three strongest numerical predictors of CAD & CAD severity are:  

1. \# Major Vessels Blocked
2. Oldpeak
3. Max Heart Rate 

Note that, based on the confidence intervals, these three could be in any order. This is only the most likely arrangment based on the data.

# Group Comparisons w/ Non-Parametric Methods

Several of the categorical variables were also strongly related to severity of CAD. However, the Spearman &rho; values are not sufficient, since the categories may not be monotonic.

In order to capture the exact magnitude and significance of these relations then, non-parametric group comparison tests such as Mann-WhitneyU and Kruskal-Wallis were used. This became necessary because, in this dataset, the severity of CAD is captured as an ordinal variable 'num,' causing certain assumptions of parametric group comparison tests (ANOVA and t-tests, e.g.) to fail.  

Concerning categorical variables, the top predictors were chest pain, thallium stress test results, and exercise-induced angina. 

## Binary Group Comparison

For exang, since there are exactly two groups and an ordinal target variable, the Mann-Whitney U test was applied, as seen below:

In [ ]:
g0 = xy.loc[xy["exang"] == 0, "num"]
g1 = xy.loc[xy["exang"] == 1, "num"]
u, p = stats.mannwhitneyu(g0, g1, alternative="two-sided")
print(f"U-value = {u}, p-value ={p: .3e}")

In order to extract the actual effect size, we can calculate the rank-biserial coefficient using the U-statistic. This provides us with a single "correlation coefficient" not dissimilar from the earlier Spearman &rho;.

In [ ]:
def r_bis(data, feature):
    groups = [
            group["num"].values
            for _, group in data.groupby(feature)
        ]
    u, _ = stats.mannwhitneyu(*groups, alternative="two-sided")
    n0,n1 = len(g0), len(g1)
    r_biser = 1 - (2*u)/(n0*n1)
    return r_biser

print(f"The exang rank-biserial coefficient is {round(r_bis(xy, 'exang'), 3)}")

Although 0.48 does represent a strong correlation, and we do have an exceptionally small p-value, it remains necessary to compute bootstrapped confidence intervals to get a sense for the total population value. We proceed via the following function:

In [ ]:
def bootstrap_r_bis(feature):
    boot = []
    for i in range(1000):
        sample = xy.sample(frac=1, replace=True)
        r_biser = r_bis(sample, feature)
        boot.append(r_biser)
        boot_series = pd.Series(boot)
    lower = boot_series.quantile(0.025)
    upper = boot_series.quantile(0.975)
    return (lower, upper, boot_series)

Applying this to exang, and graphing the results, we get:

In [ ]:
lower, upper, boot_series = bootstrap_r_bis('exang')
print(f"Our confidence interval is {(float(round(lower, 3)), float(round(upper, 3)))}")

plt.hist(boot_series, edgecolor = "black")
plt.xlabel('Rank-biserial coefficients in bootstrapped samples (exang)')
plt.ylabel('Number of incidences (n=1000)')
plt.axvline(x=lower, color = "black")
plt.axvline(x=upper, color = "black")
plt.show()

Thus, we can be 95% that there is at minimum a moderate correlation between the presence of exercise-induced angina and the severity of CAD.  

## Multi-Group Comparison

Unlike exercise-induced angina, the UCI database separates thallium stress test results (thal) and chest pain (cp) into multiple types. I will define them here:

Cp:  
1 - typical angina  
2 - atypical angina  
3 - non-anginal pain  
4 - asymptomatic  

Thal:  
3 - normal  
6 - fixed defect  
7 - reversible defect  

Instead of computing numerous pairwise comparisons, this report uses the epsilon-squared statistic to provide one value to measure the impact of each independent variable. By taking the square root of the epsilon-squared value, we can achieve a number on the same scale as the rank-biserial coefficient, allowing us to make comparisons between binary and multi-group variables.  

Below, we compute the epsilon-squared values for cp and thal:

In [ ]:
def eps_squared(data, feature):
    groups = [
            group["num"].values
            for _, group in data.groupby(feature)
        ]
    h, _ = stats.kruskal(*groups)
    n = data_points
    e_squared = h/(n-1)
    return e_squared

print(f"eps_squared (cp) = {round(eps_squared(xy, 'cp'),3)}")
print(f"eps_squared (thal) = {round(eps_squared(xy, 'thal'),3)}")


Using the following helper function, we can also compute bootstrapped confidence intervals:

In [ ]:
def bootstrap_eps_squared(feature):
    helper = []
    for j in range(1000):
        sample = xy.sample(frac=1, replace=True)
        e = eps_squared(sample, feature)
        helper.append(e)
    helper_ser = pd.Series(helper)
    lower = helper_ser.quantile(0.025)
    upper = helper_ser.quantile(0.975)
    return (lower, upper, helper_ser)

In [ ]:
strong_cats = ['thal', 'cp']
for feature in strong_cats:
    lower, upper, _ = bootstrap_eps_squared(feature)
    print(f"Epsilon-squared CI for {feature} is {(float(round(lower, 3)), float(round(upper, 3)))}")

Since both intervals are uniformly above 0.14, we can be at least 95% sure that thal and cp are strongly correlated with, and are thus important risk factors for, CAD. 

### Post-Hocs for Kruskal-Wallis

In order to measure exactly which groups differ from each other, and thus exactly how thallium stress tests and chest pain can be used to predict/detect CAD, we needed more than the epsilon-squared omnibus test. Instead, we used the post-hoc, non-parametric Dunn's test, as shown below.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
dunn_cp = posthoc_dunn(xy, 'num', 'cp', 'holm',True)
sb.heatmap(dunn_cp, annot=True, cmap='vlag', vmin=0, vmax=0.1, cbar_kws={'label': 'p-value'}, ax =ax1)
ax1.set_title("P-values for Dunn's Pairwise Comparisons (cp)")

dunn_thal = posthoc_dunn(xy, 'num', 'thal','holm',True)
sb.heatmap(dunn_thal, annot=True, cmap='vlag', vmin=0, vmax=0.1, cbar_kws={'label': 'p-value'}, ax=ax2)
ax2.set_title("P-values for Dunn's Pairwise Comparisons (thal)")

plt.tight_layout()
plt.show()


As seen above, each feature had exactly one group that differed significantly from the others (significant differences colored in blue). Thus, we can simply run multiple binary comparisons in order to determine the direction and magnitude of these differences:

In [ ]:
for i in range (1,4):
    group_a = xy[xy["cp"] == i]["num"]
    group_b = xy[xy["cp"] == 4]["num"]
    u, _ = stats.mannwhitneyu(group_a, group_b, alternative="two-sided")
    n0,n1 = len(group_a), len(group_b)
    r_biser = 1 - (2*u)/(n0*n1)
    print(f"R_bis (cp) {i}, 4 = {r_biser}")

thal_vals = [6,7]
for i in thal_vals:
    group_a = xy[xy["thal"] == i]["num"]
    group_b = xy[xy["thal"] == 3]["num"]
    u, _ = stats.mannwhitneyu(group_a, group_b, alternative="two-sided")
    n0,n1 = len(group_a), len(group_b)
    r_biser = 1 - (2*u)/(n0*n1)
    print(f"R_bis (thal) {i}, 3 = {r_biser}")

Thus, people who show other signs of CAD but are asymptomatic for chest pain have a significantly higher risk for CAD than all other groups, whereas those with no defects detected during a thallium stress test are at significantly lower risk for CAD than others.

### Categorical Forest Plot

In order to collect and graph the results, we will employ the following helper function:

In [ ]:
def build_results_cat(label, feature, row, df):
    groups = [
            group["num"].values
            for _, group in sample.groupby(feature)
        ]
    if len(groups) == 2:
        lower, upper, _ = bootstrap_r_bis(feature)
        middle = r_bis(xy, feature)
    elif len(groups) > 2:
        lower, upper, _ = bootstrap_eps_squared(feature)
        lower, upper = math.sqrt(lower), math.sqrt(upper)
        middle = math.sqrt(eps_squared(xy, feature))
    df.iloc[row] = [label, middle, lower, upper]

Now, we will build a dataframe containing the earlier rank-biserial and the square roots of the epsilon-squared values to identify a ranking of the strongest risk factors. 

In [ ]:
catvals = pd.DataFrame(columns = ['label', 'mid', 'll', 'hl'], index = [0, 1, 2])

build_results_cat("Exercise Angina", "exang", 0, catvals)
build_results_cat("TST results", "thal", 1, catvals)
build_results_cat("Chest Pain Type", 'cp', 2, catvals)

catvals = catvals.sort_values('mid')
fp.forestplot(dataframe=catvals, estimate = 'mid', ll = 'll', hl = 'hl', varlabel = 'label', xlim = (0.3, 0.7), 
              xlabel = "Effect Size", figsize= (6,3), ci_report=False, form_ci_report=False,
              **{"marker": "D", "markersize": 40, "color": "black"})
plt.tight_layout()
plt.show()

The strongest categorical predictors then, according to the data, are: 

1. Chest Pain Type
2. TST Results
3. Exercise-Induced Angina

Like earlier, this is only the most likely ordering based on the sample. In the actual population, any ordering of these three would be possible, although this remains the most probable. We can, however, claim with 95% certainty that each of these three predictors has a moderate to strong correlation with CAD, making them important risk factors.

# Predictive Machine Learning w/ Logistic Regression

In addition to statistical analysis, a logistic regression machine learning algorithm was also trained using scikit-learn to discern between the presence (num > 0) and absence (num = 0) of CAD. 

### Preprocessing  

To prevent the ML algorithm for incorrectly interpreting the categorical variables as continuous measurements, one-hot encoding wes employed on cp, thal, slope, and restecg (all of the non-binary categorical variables). Moreover, in order to standardize the numerical variables, we encoded their values with z-scores to make the eventual coefficients accurately reflect effect size.

In [ ]:
#One-hot encoding
X = pd.get_dummies(
    X,
    columns=['cp', 'thal', 'slope', 'restecg'],
    drop_first=True
)

#Standardizing numeric variables
numeric_vars = [
    'age',
    'trestbps',
    'chol',
    'thalach',
    'oldpeak',
    'ca'
]

X[numeric_vars] = StandardScaler().fit_transform(X[numeric_vars])

# Binarizing num
y_bin = (y["num"] > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y_bin, test_size=0.2, random_state=1055)

model = LogisticRegression()
model.fit(X_train, y_train)

We can test its efficiency by finding the confusion matrix, accuracy, and ROC AUC score/curve. Moreover, we can create a table of the odds ratios of each feature in the logistic regression. A value > 1 indicates a positive correlation with the target (num), and a value < 1 indicates a negative correlation. Specifically, the presence of a feature (or increase by one standard deviation in the case of numeric features) indicates a multiplication in the probability of CAD appearance by the odds ratio.

The intercept reflects the probability of CAD if all standardized variables are 0.

In [ ]:
predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]
accuracy = model.score(X_test, y_test)

print("Accuracy is:", str(round(accuracy,  3)))
print("The confusion matrix is:\n", confusion_matrix(y_test, predictions))
auc = roc_auc_score(y_test, probabilities)
print(f"The ROC AUC score is: {round(auc, 3)}")

fpr, tpr, _ = roc_curve(y_test, probabilities)

plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

odds_rates = np.round(np.exp(model.coef_), 3)
coefficients = pd.Series(odds_rates.transpose().flatten(), index=X.columns)
print(f"Coefficients:\n{coefficients}")
print("Intercept:", np.round(math.exp(model.intercept_), 3))


# Discussion

### Top Risk Factors

Through the analysis, the top numerical risk factors/predictors of CAD in the database were revealed to be 

1. Major Vessels Blocked  
2. Oldpeak  
3. Maximum Heart Rate  

and the top categorical risk factors/predictors were  

1. Fixed/Reversible Defects (TST)  
2. Chest Pain  
3. Exercise-Induced Angina  

These features were all positively correlated with CAD, except for maximum heart rate, which showed a negative correlation. 

Almost all of these correlations were as hypothesized: high oldpeak, unfavorable TST results, low maximum heart rate, and presence of exang all reflect the weakening of heart muscles due to lack of oxygen, immediately relating them to risk of CAD. Finally, the blockage of major vessels is a defining trait and cause of CAD, explaining its place as the top numerical predictor in the dataset.  

However, unintuitively, patients reported as asymptomatic for chest pain were significantly more likely to experience CAD at higher severities. Although this seems unlikely, the literature supports this finding. This painless CAD is a phenomenon called silent ischemia, and it can be indicative of more severe CAD for a number of reasons.  

1. A lack of chest pain prevents patients from taking action sooner, allowing more time for the condition to develop and worsen.
2. A lack of chest pain may indicate nerve damage, which would be associated with more severe CAD
3. A lack of chest pain may indicate adaptation by the body, which would be associated with longer-standing and thus more severe CAD

(Shams et. al 2024)

### Predictive Modeling

In the end, the logistic regression was able to achieve an 88% accuracy with a 0.94 ROC AUC score, which is generally considered to be outstanding performance. The algorithm is also capable of calculating a percentage risk for any patients given data for these 13 features, which could be helpful in diagnostic screening. Particularly, many of the features with high odds ratios are simple to measure independently, meaning that at-risk individuals may be able tl accurately assess their probability of CAD without the aid of a medical may professional.  

The odds ratios found by the predictive modeling also mostly support the conclusions drawn by the earlier univariate nonparametric analysis.

The strongest categorical predictors of CAD are indeed TST results and chest pain absence, although exang was found to be weaker than expected while sex was stronger. By binarizing the CAD data before computing rank-biserial values, it was found that, although exang is a superior predictor of the severity of CAD, sex is a better predictor of its presence.

Moreover, the best numerical predictors for CAD in the univariate and multivariate analysis line up perfectly, with thalach, ca, and oldpeak as the top three.

### Limitations

One major limitation of this project is the relatively small sample size. With only 300 patients, the bootstrapped confidence intervals remain large, and even assuming perfectly unbiased sampling, we can only be 95% sure that the true correlational value for the larger population even lies in the interval at all,

Moreover, this dataset is outdated. It was originally published in 1988, marking 38 years since its inception. Perhaps the biggest sign of its age is the inclusion of the thallium stress test, which is replaced by technetium today because of its lower radioactivity leading to less dangerous radiation exposure and clearer images (Nuclear Stress Test 2025). The age of the dataset may also lead to results that are inaccurate today due to changing technology and social trends. For example, asymptomatic chest pain may be a weaker predictor of CAD severity today since even without chest pain as an indicator, CAD is more likely to be caught earlier with more modern screening methods.

A final limitation of this project is that the data is collected exclusively from cardiac patients in Cleveland, Ohio. Such a sample may not be capable of reflecting trends in other parts of America, let alone on a global scale. For example, patients in rural areas are less likely to seek routine medical care, potentially making silent ischemia (lack of chest pain) an even greater predictor of CAD (Spleen et. al 2015).

### Future Directions

With the help of future research, many of these limitations can be resolved. One clear direction future research could take is the usage of a more modern database capable of reflecting modern diagnostic methods and social trends the UCI database is incapable of capturing. In the same vein, using a database from a different location, perhaps specifically a rural area, would also help reveal a more accurate picture of data trends occurring over the larger population. Finally, simply using a larger sample size could help shrink confidence intervals and provide a more refined predictive modeling algorithm

# Conclusion

This project analyzed 13 features in order to investigate potential predictors of the presence and severity of coronary artery disease (CAD). 

Top categorical and numerical predictors were found using nonparametric statistical tests like Mann-Whitney U and Kruskal-Wallis. Uncertainty was estimated using bootstrapped confidence intervals.

These findings were supported by logistic regression classification modeling, which achieved an 87% accuracy and a 0.94 ROC AUC score.

Overall, this project illustrates the value of combining statistical inference, uncertainty estimation, and predictive modeling when analyzing healthcare data.

# References

1. Janosi, A., Steinbrunn, W., Pfisterer, M., & Detrano, R. (1989). Heart Disease [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C52P4X
2. Shams P, Gul Z, Makaryus AN. Silent Myocardial Ischemia. [Updated 2024 Mar 7]. In: StatPearls [Internet]. Treasure Island (FL): StatPearls Publishing; 2026 Jan-. Available from: [https://www.ncbi.nlm.nih.gov/books/NBK536915/](https://www.ncbi.nlm.nih.gov/books/NBK536915/)
3. Wackers FJ. Comparison of thallium-201 and technetium-99m methoxyisobutyl isonitrile. Am J Cardiol. 1992 Nov 5;70(14):30E-34E. Available from: [https://doi.org/10.1016/0002-9149(92)90036-X](https://doi.org/10.1016/0002-9149(92)90036-X)
3. Spleen AM, Lengerich EJ, Camacho FT, Vanderpool RC. Health care avoidance among rural populations: results from a nationally representative survey. J Rural Health. 2014;30:79–88. Available from: https://doi.org/10.1111/jrh.12032